# Mystery & Sci-Fi — One-Click Colab Run

This is the straight-through runner for the six-scene Mystery & Sci-Fi production. It does the setup, syncs the latest production code, checks your Colab Secret, prepares the episode, writes narration and timed captions, and creates the Google Flow manifest in one cell.

**Use:** Open in Google Colab and run the single code cell. Choose **R** to resume an episode or **N** to enter a new premise. When resuming, press Enter or type **YES** to accept the displayed saved project ID. New and resumed episodes share the same monthly Flow credit ledger.

- Uses your existing Google Drive workspace and resumes files from the previous attempt.
- Uses the six-scene premise already selected.
- Keeps separately billed Gemini Developer API image/video calls **off**. Flow image creation uses the separate Pro daily image quota.
- Uses Veo 3.1 Lite at 8 seconds (10 Flow credits per scene) and one output per request.
- Plans for 15 six-scene episodes from the 1,000 monthly credit pool after a 100-credit reserve; 20 episodes are possible by using six separate 50-credit daily refreshes across the month.
- Checks the signed-in Flow balance before each batch, protects the reserve, stores a Drive-based monthly ledger and browser profile, skips completed scenes, and stops if model/duration/credit verification fails.
- The first run may require you to sign in normally in the embedded Flow browser. The automation handles image creation, uploads, prompts, video generation, downloads, and final assembly; it does not publish to YouTube.


In [ ]:
import os, sys, json, time, shutil, subprocess, traceback, html as html_lib, re
from pathlib import Path
from google.colab import drive, userdata
from IPython.display import display, Markdown, HTML

# Fixed for the first six-scene test; edit these values only when making a different episode.
TOPIC = "An abandoned coastal radio station receives a warning about an event that has not happened yet."
SCENE_COUNT = 6
VOICE = "en-US-ChristopherNeural"
WHISPER_MODEL = "base.en"
WHISPER_DEVICE = "auto"
WHISPER_COMPUTE_TYPE = "int8"
WHISPER_DEVICE_PREFERENCE = "auto"
# Google Flow automation and monthly credit controls.
FLOW_VIDEO_MODEL = "Veo 3.1 - Lite"       # 10 credits for an 8-second 720p scene on Pro
FLOW_IMAGE_MODEL = "Nano Banana Pro"   # Uses a separate daily image quota
FLOW_DURATION_SECONDS = 8
FLOW_MAX_SCENES_PER_RUN = 6            # Six scenes = 60 video credits per episode
FLOW_MAX_IMAGES_PER_RUN = 25           # Includes character references, frames and thumbnails
FLOW_MONTHLY_CREDITS = 1000
FLOW_DAILY_CREDITS = 50
FLOW_RESERVE_PERCENT = 10
FLOW_TARGET_VIDEOS = 15                # 15 six-scene episodes = 900 credits
FLOW_STRETCH_VIDEOS = 20               # Requires six separate daily 50-credit refreshes
# Keep this stable for the current Google AI Pro Flow billing cycle.
# Change it only after Google confirms the next monthly credit refresh.
FLOW_CREDIT_CYCLE_ID = "pro-cycle-2026-11"

PROJECT_ID = "20261120_123020_mystery_scifi_An_abandoned_coastal_radio_station_recei"
RESUME_PROJECT_ID = PROJECT_ID
REPO_URL = "https://github.com/topboyasian-stack/gta6-autoclipper.git"
ROOT = Path("/content/gta6-autoclipper")
LOG_PATH = None

try:
    # Validate the key before Drive mounting, cloning, package installation, or API calls.
    try:
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        api_key = None
    api_key = str(api_key or os.environ.get("GEMINI_API_KEY", "")).strip()
    if len(api_key) < 10:
        raise RuntimeError(
            "GEMINI_API_KEY is missing or inaccessible. Add it under Colab Secrets "
            "with notebook access enabled, or provide it in the environment. "
            "The run was stopped before production work or API calls."
        )

    if not Path("/content/drive/MyDrive").is_dir():
        print("Connecting Google Drive…", flush=True)
        drive.mount("/content/drive")

    WORKSPACE = Path("/content/drive/MyDrive/GTA6_AutoClipper")
    PROJECTS_ROOT = WORKSPACE / "projects"
    PROJECTS_ROOT.mkdir(parents=True, exist_ok=True)
    ACTIVE_PROJECT_POINTER = WORKSPACE / "last_mystery_project.json"
    default_resume_id = RESUME_PROJECT_ID
    if ACTIVE_PROJECT_POINTER.is_file():
        try:
            pointer = json.loads(ACTIVE_PROJECT_POINTER.read_text(encoding="utf-8"))
            default_resume_id = str(pointer.get("project_id") or default_resume_id)
        except Exception as pointer_error:
            print("WARNING: Could not read last-project pointer; using the notebook's original project ID. "
                  + str(pointer_error), flush=True)

    print("\nMystery & Sci-Fi production selector")
    print("  R = resume an existing episode (no completed scene is regenerated)")
    print("  N = start a new episode and add its Flow spend to the same monthly ledger")
    run_choice = input("Choose R or N [R]: ").strip().lower() or "r"
    if run_choice in {"n", "new"}:
        TOPIC = input("Enter the new episode premise: ").strip()
        if len(TOPIC) < 8:
            raise ValueError("A new episode premise must contain at least 8 characters.")
        slug = re.sub(r"[^a-z0-9]+", "_", TOPIC.casefold()).strip("_")[:42] or "episode"
        PROJECT_ID = "mystery_scifi_" + time.strftime("%Y%m%d_%H%M%S") + "_" + slug
        RESUME_PROJECT_ID = PROJECT_ID
    elif run_choice in {"r", "resume"}:
        resume_answer = input(
            "Project ID to resume [" + default_resume_id + "] (type YES to accept): "
        ).strip()
        if resume_answer.casefold() in {"yes", "y"}:
            RESUME_PROJECT_ID = default_resume_id
        else:
            RESUME_PROJECT_ID = resume_answer or default_resume_id
        if not re.fullmatch(r"[A-Za-z0-9_-]{8,120}", RESUME_PROJECT_ID):
            raise ValueError(
                "Invalid project ID. Press Enter or type YES to use the displayed project, "
                "or enter a valid saved Project ID."
            )
        PROJECT_ID = RESUME_PROJECT_ID
    else:
        raise ValueError("Choose R to resume or N to start a new episode.")

    FLOW_EXPECTED_GOOGLE_EMAIL = input(
        "Exact Google account email that should own/use the Flow credits for this run: "
    ).strip()
    if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", FLOW_EXPECTED_GOOGLE_EMAIL):
        raise ValueError(
            "A valid target Google account email is required. Run stopped before browser setup or generation."
        )
    print(f"Google Flow account required for this run: {FLOW_EXPECTED_GOOGLE_EMAIL}", flush=True)
    print(
        "The Flow worker must verify this exact email in the active Google account UI before reading "
        "the balance or submitting any image/video generation.",
        flush=True,
    )

    PROJECT_DIR = PROJECTS_ROOT / RESUME_PROJECT_ID
    OUTPUT_DIR = PROJECT_DIR / "mystery_scifi"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    LOG_PATH = OUTPUT_DIR / "mystery_run.log"

    def note(message):
        stamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{stamp}] {message}"
        print(line, flush=True)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write(line + "\n")
            log.flush()

    with LOG_PATH.open("a", encoding="utf-8") as log:
        log.write("\n\n===== One-click six-scene run: " + time.strftime("%Y-%m-%d %H:%M:%S") + " =====\n")

    note("Persistent log: " + str(LOG_PATH))
    note("Project output: " + str(OUTPUT_DIR))
    ACTIVE_PROJECT_POINTER.write_text(
        json.dumps({"project_id": RESUME_PROJECT_ID, "output_dir": str(OUTPUT_DIR),
                    "updated_at": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=2),
        encoding="utf-8",
    )

    def run_logged(label, command, env=None):
        note("START: " + label)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write("COMMAND: " + " ".join(str(part) for part in command) + "\n")
            log.flush()
            process = subprocess.Popen(
                [str(part) for part in command],
                cwd=str(ROOT) if ROOT.is_dir() else None,
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                errors="replace",
            )
            log.write("PROCESS_ID: " + str(process.pid) + "\n")
            log.flush()
            print("Process started: " + str(process.pid), flush=True)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
            log.write("EXIT_CODE: " + str(return_code) + "\n")
            log.flush()
        if return_code != 0:
            raise RuntimeError(f"{label} failed with exit code {return_code}. See {LOG_PATH}")
        note("DONE: " + label)

    def topic_tokens(value):
        import re
        ignored = {
            "a", "an", "and", "are", "as", "at", "by", "for", "from", "in",
            "into", "is", "it", "of", "on", "or", "the", "that", "this", "to",
            "was", "were", "with",
        }
        return {
            token for token in re.findall(r"[a-z0-9]+", str(value or "").casefold())
            if len(token) > 1 and token not in ignored
        }

    def topics_differ_significantly(saved_topic, requested_topic):
        import re
        saved_normalized = " ".join(re.findall(r"[a-z0-9]+", str(saved_topic or "").casefold()))
        requested_normalized = " ".join(re.findall(r"[a-z0-9]+", str(requested_topic or "").casefold()))
        if not saved_normalized or not requested_normalized or saved_normalized == requested_normalized:
            return False
        saved_tokens = topic_tokens(saved_topic)
        requested_tokens = topic_tokens(requested_topic)
        if not saved_tokens or not requested_tokens:
            return True
        similarity = len(saved_tokens & requested_tokens) / max(1, len(saved_tokens | requested_tokens))
        return similarity < 0.30

    if (ROOT / ".git").is_dir():
        run_logged("Sync latest GitHub production code", ["git", "-C", str(ROOT), "fetch", "--depth=1", "origin", "main"])
        run_logged("Select the latest tested main commit", ["git", "-C", str(ROOT), "reset", "--hard", "FETCH_HEAD"])
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        run_logged("Download production code", ["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])

    # The clone is executed in a subprocess; that does not add its directory to this
    # notebook kernel's sys.path. Add it explicitly before importing project modules.
    if not ROOT.is_dir():
        raise FileNotFoundError(f"Production repository was not downloaded: {ROOT}")
    required_flow_files = (
        ROOT / "scripts" / "__init__.py",
        ROOT / "scripts" / "colab_flow_display.py",
        ROOT / "scripts" / "flow_credits.py",
    )
    missing_flow_files = [str(path) for path in required_flow_files if not path.is_file()]
    if missing_flow_files:
        raise FileNotFoundError("Required Flow automation files are missing: " + ", ".join(missing_flow_files))
    ROOT_IMPORT_PATH = str(ROOT.resolve())
    if ROOT_IMPORT_PATH in sys.path:
        sys.path.remove(ROOT_IMPORT_PATH)
    sys.path.insert(0, ROOT_IMPORT_PATH)
    import importlib
    importlib.invalidate_caches()
    note("Configured notebook Python import path: " + ROOT_IMPORT_PATH)

    # Resume takes priority over the newly entered TOPIC when the saved project is
    # materially different. Warn first and preserve that plan; never delete it by count.
    plan_path = OUTPUT_DIR / "episode.json"
    saved_plan_topic = None
    resume_topic_override = False
    if plan_path.is_file():
        try:
            previous = json.loads(plan_path.read_text(encoding="utf-8"))
            saved_plan_topic = str(previous.get("topic", "")).strip() or None
            previous_count = len(previous.get("scenes", []))
        except Exception as parse_error:
            previous_count = -1
            note(f"WARNING: Existing episode.json could not be read ({type(parse_error).__name__}: {parse_error}); it will be backed up before a new plan is generated.")
        if saved_plan_topic and topics_differ_significantly(saved_plan_topic, TOPIC):
            resume_topic_override = True
            note(
                f"WARNING: RESUME_PROJECT_ID='{RESUME_PROJECT_ID}' is overriding TOPIC. "
                f"Saved project plan topic: {saved_plan_topic!r}. Current TOPIC: {TOPIC!r}. "
                "The existing project's plan will be reused unchanged."
            )
        elif previous_count != SCENE_COUNT:
            backup = OUTPUT_DIR / ("episode.backup.before_scene_count_change_" + time.strftime("%Y%m%d_%H%M%S") + ".json")
            shutil.copy2(plan_path, backup)
            plan_path.unlink()
            saved_plan_topic = None
            note(f"Existing plan had {previous_count} scenes; backed it up to {backup} before generating the requested {SCENE_COUNT}-scene plan.")
        else:
            note("Found a saved plan with the requested topic and scene count; it will be reused.")

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / "project_state.json").write_text(
        json.dumps({
            "project_name": RESUME_PROJECT_ID,
            "mode": "Mystery & Sci-Fi Automation",
            "story_title": (saved_plan_topic or TOPIC)[:160],
            "mystery_output": str(OUTPUT_DIR),
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "setup": "one-click-colab",
        }, indent=2),
        encoding="utf-8",
    )

    if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
        run_logged(
            "Install FFmpeg",
            ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core"],
        )
    else:
        note("FFmpeg and ffprobe are available.")

    requirements_path = ROOT / "requirements.txt"
    if not requirements_path.is_file():
        raise FileNotFoundError(
            f"Repository requirements file is missing: {requirements_path}. "
            "The notebook will not fall back to unpinned ad-hoc pip installs."
        )
    note("Install pinned production dependencies from the repository requirements.txt.")
    run_logged(
        "Install pinned Python dependencies",
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--disable-pip-version-check", "-r", str(requirements_path),
        ],
    )

    # Probe host GPU visibility, then verify that the installed CTranslate2 build
    # can actually use CUDA. A detected host GPU alone is not enough.
    gpu_signal = False
    gpu_signal_source = []
    try:
        nvidia_probe = subprocess.run(
            ["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10, check=False
        )
        if nvidia_probe.returncode == 0 and nvidia_probe.stdout.strip():
            gpu_signal = True
            gpu_signal_source.append("nvidia-smi")
    except (FileNotFoundError, subprocess.SubprocessError):
        pass
    try:
        import torch
        if torch.cuda.is_available():
            gpu_signal = True
            gpu_signal_source.append("torch.cuda.is_available")
    except Exception:
        pass
    ctranslate_cuda_count = 0
    try:
        import ctranslate2
        ctranslate_cuda_count = int(ctranslate2.get_cuda_device_count())
        if ctranslate_cuda_count > 0:
            gpu_signal = True
            gpu_signal_source.append("CTranslate2")
    except Exception as gpu_error:
        note(f"WARNING: CTranslate2 CUDA probe failed; CPU fallback will be used. {type(gpu_error).__name__}: {gpu_error}")

    if gpu_signal and ctranslate_cuda_count > 0:
        WHISPER_DEVICE = "cuda"
        WHISPER_COMPUTE_TYPE = "float16"
        note(
            f"Whisper GPU detected ({', '.join(gpu_signal_source)}; "
            f"{ctranslate_cuda_count} CTranslate2 CUDA device(s)); "
            "preferred device=cuda, compute_type=float16."
        )
    else:
        WHISPER_DEVICE = "cpu"
        WHISPER_COMPUTE_TYPE = "int8"
        if gpu_signal and ctranslate_cuda_count < 1:
            note("WARNING: A GPU signal exists, but CTranslate2 cannot use CUDA; falling back to CPU int8.")
        else:
            note("No usable CUDA GPU detected; Whisper will use CPU int8.")
    note("Whisper device preference is 'auto'; a CUDA initialization failure will retry on CPU int8.")

    # Keep the legacy project variable synchronized with the explicit resume selector.
    PROJECT_ID = RESUME_PROJECT_ID
    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "GEMINI_API_KEY": api_key,
        "ALLOW_PAID_IMAGE_API": "false",
        "IMAGE_PROVIDER": "manual_flow",
        "VIDEO_PROVIDER": "manual_flow",
        "ENABLE_PAID_VIDEO_API": "0",
        "WHISPER_DEVICE": WHISPER_DEVICE,
        "WHISPER_COMPUTE_TYPE": WHISPER_COMPUTE_TYPE,
        "MYSTERY_WHISPER_DEVICE": WHISPER_DEVICE_PREFERENCE,
        "MYSTERY_WHISPER_MODEL": WHISPER_MODEL,
        "RESUME_PROJECT_ID": PROJECT_ID,
        "MYSTERY_NARRATOR_VOICE": VOICE,
        "MYSTERY_MUSIC_TRACK": str(OUTPUT_DIR / "no_music_selected.mp3"),
    })
    command = [
        sys.executable, str(ROOT / "main.py"),
        "--stage", "prepare",
        "--topic", TOPIC,
        "--scene-count", str(SCENE_COUNT),
        "--output", str(OUTPUT_DIR),
        "--voice", VOICE,
        "--whisper-model", WHISPER_MODEL,
        "--whisper-device", WHISPER_DEVICE_PREFERENCE,
        "--max-scenes-per-flow-run", "10",
    ]
    # Deliberately omit --allow-paid-images.
    # On resume, keep completed preparation artifacts and avoid re-running the CPU-heavy
    # narration/Whisper stages after a later Flow/browser setup failure.
    episode_path = OUTPUT_DIR / "episode.json"
    reusable_preparation = False
    if episode_path.is_file() and episode_path.stat().st_size > 0:
        try:
            cached_episode = json.loads(episode_path.read_text(encoding="utf-8"))
            cached_scene_count = len(cached_episode.get("scenes", []))
            cached_required = [
                episode_path,
                OUTPUT_DIR / "episode_brief.md",
                OUTPUT_DIR / "assets_generation_plan.json",
                OUTPUT_DIR / "assets_generation_plan.md",
                OUTPUT_DIR / "audio" / "narration_manifest.json",
                OUTPUT_DIR / "flow_upload_manifest.json",
                OUTPUT_DIR / "flow_upload_manifest.txt",
                OUTPUT_DIR / "flow_master_prompt.md",
            ]
            cached_missing = [
                path for path in cached_required
                if not path.is_file() or path.stat().st_size == 0
            ]
            for scene_number in range(1, cached_scene_count + 1):
                scene_id = f"scene_{scene_number:03d}"
                for suffix in (".mp3", ".wav"):
                    path = OUTPUT_DIR / "audio" / f"{scene_id}{suffix}"
                    if not path.is_file() or path.stat().st_size == 0:
                        cached_missing.append(path)
            cached_captions = (
                OUTPUT_DIR / "captions" / "subtitles.srt",
                OUTPUT_DIR / "captions" / "subtitles.ass",
            )
            if not any(path.is_file() and path.stat().st_size > 0 for path in cached_captions):
                cached_missing.append(OUTPUT_DIR / "captions" / "subtitles.srt")
            scene_count_valid = (
                cached_scene_count > 0
                and (resume_topic_override or cached_scene_count == SCENE_COUNT)
            )
            reusable_preparation = scene_count_valid and not cached_missing
            if reusable_preparation:
                note(
                    "Reusing verified preparation artifacts for "
                    + str(cached_scene_count)
                    + " scenes; narration and Whisper captions will not be regenerated."
                )
            else:
                note(
                    "Saved preparation is incomplete; missing/invalid artifacts: "
                    + ", ".join(str(path) for path in cached_missing[:12])
                )
        except Exception as cache_error:
            note(
                "Could not validate saved preparation; rebuilding preparation safely. "
                + type(cache_error).__name__ + ": " + str(cache_error)
            )
    if not reusable_preparation:
        run_logged(
            "Prepare six-scene episode, narration, captions, and Flow manifest",
            command,
            env=env,
        )

    if not episode_path.is_file() or episode_path.stat().st_size == 0:
        raise RuntimeError(f"Required episode plan is missing or empty: {episode_path}")
    episode = json.loads(episode_path.read_text(encoding="utf-8"))
    actual_scene_count = len(episode.get("scenes", []))
    if actual_scene_count < 1:
        raise RuntimeError("episode.json contains no scenes.")
    if not resume_topic_override and actual_scene_count != SCENE_COUNT:
        raise RuntimeError(f"Expected {SCENE_COUNT} scenes, found {actual_scene_count}.")

    # Do not declare SUCCESS until each required deliverable exists and is non-empty.
    required_files = [
        OUTPUT_DIR / "episode.json",
        OUTPUT_DIR / "episode_brief.md",
        OUTPUT_DIR / "assets_generation_plan.md",
        OUTPUT_DIR / "audio" / "narration_manifest.json",
        OUTPUT_DIR / "flow_upload_manifest.json",
        OUTPUT_DIR / "flow_upload_manifest.txt",
        OUTPUT_DIR / "flow_master_prompt.md",
    ]
    missing_or_empty = [
        str(path) for path in required_files
        if not path.is_file() or path.stat().st_size == 0
    ]
    if missing_or_empty:
        raise RuntimeError(
            "Preparation finished but required artifacts are missing or empty: "
            + ", ".join(missing_or_empty)
        )

    audio_dir = OUTPUT_DIR / "audio"
    missing_audio = []
    for scene_number in range(1, actual_scene_count + 1):
        scene_id = f"scene_{scene_number:03d}"
        for suffix in (".mp3", ".wav"):
            audio_path = audio_dir / f"{scene_id}{suffix}"
            if not audio_path.is_file() or audio_path.stat().st_size == 0:
                missing_audio.append(str(audio_path))
    if missing_audio:
        raise RuntimeError(
            "Narration audio is incomplete; these expected per-scene files are missing or empty: "
            + ", ".join(missing_audio)
        )

    captions_srt = OUTPUT_DIR / "captions" / "subtitles.srt"
    captions_ass = OUTPUT_DIR / "captions" / "subtitles.ass"
    if not any(path.is_file() and path.stat().st_size > 0 for path in (captions_srt, captions_ass)):
        raise RuntimeError("Caption generation did not produce a non-empty subtitles.srt or subtitles.ass file.")

    # Record the plan actually used, not the TOPIC value when a resume override occurred.
    state_topic = str(episode.get("topic") or saved_plan_topic or TOPIC)
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / "project_state.json").write_text(
        json.dumps({
            "project_name": RESUME_PROJECT_ID,
            "mode": "Mystery & Sci-Fi Automation",
            "story_title": state_topic[:160],
            "mystery_output": str(OUTPUT_DIR),
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "setup": "one-click-colab",
            "resume_topic_override": resume_topic_override,
            "whisper_device": WHISPER_DEVICE,
            "whisper_compute_type": WHISPER_COMPUTE_TYPE,
            "paid_image_api": False,
            "image_provider": "google_flow_browser",
            "video_provider": "google_flow_browser",
            "paid_video_api": False,
        }, indent=2),
        encoding="utf-8",
    )

    note("SUCCESS: every required episode, narration, caption, and Google Flow artifact exists and is non-empty.")
    # Visual production is now handled by a persistent, signed-in Google Flow browser.
    # No Gemini Developer API image/video billing is enabled.
    note("Starting automated Google Flow production with live balance and reserve checks.")
    flow_requirements = ROOT / "requirements-flow.txt"
    if not flow_requirements.is_file():
        raise FileNotFoundError(f"Optional Flow browser requirements are missing: {flow_requirements}")
    run_logged(
        "Install pinned Playwright Flow browser dependency",
        [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", str(flow_requirements)],
    )
    run_logged(
        "Install Colab browser display utilities",
        ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq xvfb x11vnc novnc websockify"],
    )
    run_logged(
        "Install temporary browser gateway",
        ["bash", "-lc", "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nginx"],
    )
    run_logged(
        "Install Chromium for Google Flow",
        [sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"],
    )

    flow_display = None
    flow_gateway = None
    flow_tunnel = None
    try:
        # Load the current Colab display helper from the refreshed repository source.
        flow_helper_path = ROOT / "scripts" / "colab_flow_display.py"
        flow_helper_spec = importlib.util.spec_from_file_location(
            "_mystery_colab_flow_display_runtime", flow_helper_path
        )
        if flow_helper_spec is None or flow_helper_spec.loader is None:
            raise RuntimeError(f"Could not load Flow display helper from {flow_helper_path}")
        flow_helper_module = importlib.util.module_from_spec(flow_helper_spec)
        flow_helper_spec.loader.exec_module(flow_helper_module)
        ColabFlowDisplay = flow_helper_module.ColabFlowDisplay
        note("Loaded Flow display helper directly from refreshed source: " + str(flow_helper_path))

        flow_display = ColabFlowDisplay()
        flow_display_info = flow_display.start()
        os.environ["DISPLAY"] = flow_display_info["display"]
        flow_env = os.environ.copy()
        flow_env.update(env)
        flow_env["DISPLAY"] = flow_display_info["display"]

        # Cloudflare's email-protected Quick Tunnel currently has a known auth-session
        # expiry bug. Use a 256-bit one-time URL path at the local nginx gateway instead.
        gateway_helper_path = ROOT / "scripts" / "colab_flow_gateway.py"
        gateway_helper_spec = importlib.util.spec_from_file_location(
            "_mystery_colab_flow_gateway_runtime", gateway_helper_path
        )
        if gateway_helper_spec is None or gateway_helper_spec.loader is None:
            raise RuntimeError(f"Could not load browser gateway helper from {gateway_helper_path}")
        gateway_helper_module = importlib.util.module_from_spec(gateway_helper_spec)
        gateway_helper_spec.loader.exec_module(gateway_helper_module)
        ColabFlowGateway = gateway_helper_module.ColabFlowGateway
        build_viewer_url = gateway_helper_module.build_viewer_url
        flow_gateway = ColabFlowGateway(
            upstream_port=flow_display_info["web_port"],
            listen_port=6081,
        )
        gateway_info = flow_gateway.start()
        note("Temporary local browser gateway is ready; requests without its random access path are rejected.")

        tunnel_helper_path = ROOT / "scripts" / "colab_flow_tunnel.py"
        tunnel_helper_spec = importlib.util.spec_from_file_location(
            "_mystery_colab_flow_tunnel_runtime", tunnel_helper_path
        )
        if tunnel_helper_spec is None or tunnel_helper_spec.loader is None:
            raise RuntimeError(f"Could not load browser tunnel helper from {tunnel_helper_path}")
        tunnel_helper_module = importlib.util.module_from_spec(tunnel_helper_spec)
        tunnel_helper_spec.loader.exec_module(tunnel_helper_module)
        ColabFlowTunnel = tunnel_helper_module.ColabFlowTunnel
        flow_tunnel = ColabFlowTunnel(
            origin_url=f"http://127.0.0.1:{gateway_info['port']}",
            log_path=OUTPUT_DIR / "flow_tunnel.log",
        )
        tunnel_url = flow_tunnel.start()
        token_path = str(gateway_info["path_prefix"])
        # noVNC path is absolute; a relative path duplicates the token because
        # vnc.html already lives inside this gateway prefix.
        viewer_url = build_viewer_url(tunnel_url, token_path)
        display(Markdown(
            "## Temporary Google Flow browser\n\n"
            f"**[Open the live browser desktop]({viewer_url})**\n\n"
            "This is a private, temporary capability link protected by a random 256-bit path. "
            "Anyone who has the complete link can access this browser while the Colab cell is running, "
            "so do not forward or post it. The link and its access path are removed when the cell stops. "
            "No Cloudflare email login is used.\n\n"
            f"Sign in to Google inside the browser as **{FLOW_EXPECTED_GOOGLE_EMAIL}**. "
            "The worker verifies that exact Google account before checking credits or generating anything.\n\n"
            f"**Model:** {FLOW_VIDEO_MODEL} · **Duration:** {FLOW_DURATION_SECONDS}s · "
            f"**Run cap:** {FLOW_MAX_SCENES_PER_RUN} scenes · **Protected reserve:** "
            f"{FLOW_MONTHLY_CREDITS * FLOW_RESERVE_PERCENT // 100} credits"
        ))
        note("Temporary browser link is ready. It is not written to the persistent run log.")

        flow_command = [
            sys.executable, str(ROOT / "scripts" / "flow_automator.py"),
            "--output", str(OUTPUT_DIR),
            "--workspace", str(WORKSPACE),
            "--profile-dir", str(WORKSPACE / "browser_profiles" / "google_flow"),
            "--expected-account", FLOW_EXPECTED_GOOGLE_EMAIL,
            "--credit-ledger", str(WORKSPACE / "flow_credit_ledger.json"),
            "--cycle-id", FLOW_CREDIT_CYCLE_ID,
            "--monthly-credits", str(FLOW_MONTHLY_CREDITS),
            "--daily-credits", str(FLOW_DAILY_CREDITS),
            "--reserve-percent", str(FLOW_RESERVE_PERCENT),
            "--target-videos", str(FLOW_TARGET_VIDEOS),
            "--stretch-videos", str(FLOW_STRETCH_VIDEOS),
            "--video-model", FLOW_VIDEO_MODEL,
            "--image-model", FLOW_IMAGE_MODEL,
            "--duration-seconds", str(FLOW_DURATION_SECONDS),
            "--max-scenes", str(FLOW_MAX_SCENES_PER_RUN),
            "--max-images", str(FLOW_MAX_IMAGES_PER_RUN),
        ]
        run_logged("Generate Flow starting frames and animate all six scenes", flow_command, env=flow_env)

        flow_report_path = OUTPUT_DIR / "flow_automation_report.json"
        if not flow_report_path.is_file() or flow_report_path.stat().st_size == 0:
            raise RuntimeError(f"Flow automation did not write its report: {flow_report_path}")
        flow_report = json.loads(flow_report_path.read_text(encoding="utf-8"))
        missing_clips = flow_report.get("clips_missing", [])
        if not missing_clips:
            run_logged(
                "Assemble final 1080p episode with narration and subtitles",
                [
                    sys.executable, str(ROOT / "main.py"),
                    "--stage", "assemble",
                    "--output", str(OUTPUT_DIR),
                ],
                env=flow_env,
            )
            final_video = OUTPUT_DIR / "final_youtube_video.mp4"
            if not final_video.is_file() or final_video.stat().st_size < 100_000:
                raise RuntimeError(f"Final episode export is missing or too small: {final_video}")
            note("SUCCESS: complete episode rendered and assembled: " + str(final_video))
        else:
            note(
                "Flow automation checkpoint: "
                + str(flow_report.get("clips_complete", 0)) + "/" + str(flow_report.get("scene_count", 0))
                + " clips complete. Missing scenes: " + ", ".join(missing_clips)
                + ". Re-run this same notebook to resume; completed frames and clips are skipped."
            )
    finally:
        if flow_tunnel is not None:
            flow_tunnel.stop()
        if flow_gateway is not None:
            flow_gateway.stop()
        if flow_display is not None:
            flow_display.stop()

    display(Markdown(
        "## Episode preparation complete\n\n"
        f"**Episode:** {episode.get('title', 'Untitled')}\n\n"
        f"**Plan topic actually used:** {state_topic}\n\n"
        f"**Scenes:** {actual_scene_count}\n\n"
        f"**Resume project:** {RESUME_PROJECT_ID}\n\n"
        f"**Project folder:** {OUTPUT_DIR}\n\n"
        f"**Flow manifest:** {OUTPUT_DIR / 'flow_upload_manifest.txt'}\n\n"
        f"**Master prompt:** {OUTPUT_DIR / 'flow_master_prompt.md'}\n\n"
        f"**Narration audio:** {audio_dir} (per-scene MP3 and WAV files)\n\n"
        f"**Captions:** {captions_srt if captions_srt.is_file() else captions_ass}\n\n"
        f"**Whisper:** {WHISPER_DEVICE} / {WHISPER_COMPUTE_TYPE} (automatic CPU fallback enabled)\n\n"
        f"**Persistent run log:** {LOG_PATH}\n\n"
        "Gemini Developer API image/video calls remain disabled. Google Flow is automated through the persistent browser worker with a live credit ledger."
    ))

    print("Flow prompt/audit files: " + str(OUTPUT_DIR / "flow_master_prompt.md"))
    print("Flow upload manifest: " + str(OUTPUT_DIR / "flow_upload_manifest.json"))
    print("Flow credit ledger: " + str(WORKSPACE / "flow_credit_ledger.json"))

    print("\nVerified artifact checklist:")
    for path in required_files:
        print(f"  [OK] {path} ({path.stat().st_size:,} bytes)")
    print(f"  [OK] {actual_scene_count} MP3 narration files and {actual_scene_count} WAV audio files")
    print(f"  [OK] Caption file: {captions_srt if captions_srt.is_file() else captions_ass}")
    print("  [OK] Flow prompt and manifest are saved in Drive for audit and resume.")
    print("Google Drive project folder: " + str(PROJECT_DIR))
    print("Episode output folder: " + str(OUTPUT_DIR))
    print("Persistent run log: " + str(LOG_PATH))

except Exception as exc:
    error = f"FAILED: {type(exc).__name__}: {exc}"
    print("\n" + error, flush=True)
    if LOG_PATH is not None:
        try:
            with LOG_PATH.open("a", encoding="utf-8") as log:
                log.write("\n" + error + "\n")
                log.write(traceback.format_exc() + "\n")
                log.flush()
            tail = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-35:]
            print("\nLast log lines:", flush=True)
            print("\n".join(tail), flush=True)
            print("\nFull persistent log: " + str(LOG_PATH), flush=True)
        except Exception as log_error:
            print("Could not append to the project log: " + str(log_error), flush=True)
            print(traceback.format_exc(), flush=True)
    else:
        print("The failure occurred before the project log was initialized.", flush=True)
        print(traceback.format_exc(), flush=True)
